# Let's go PRO!

Advanced RAG Techniques!

Let's start by digging into ingest:

1. No LangChain! Just native for maximum flexibility
2. Let's use an LLM to divide up chunks in a sensible way
3. Let's use the best chunk size and encoder from yesterday
4. Let's also have the LLM rewrite chunks in a way that's most useful ("document pre-processing")

In [ ]:
from pathlib import Path
from openai import AzureOpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import os
from App.config import azure_endpoint, api_version, headers
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv('cd_api_key_backup')

# MODEL = 'azure/gpt-4.1-mini'
MODEL = "azure/gpt-4.1-nano"

DB_NAME = "week5/preprocessed_db"
collection_name = "week5/docs"
embedding_model = "all-MiniLM-L6-v2" # need to update
KNOWLEDGE_BASE_PATH = Path("week5/knowledge-base")
AVERAGE_CHUNK_SIZE = 500

openai = AzureOpenAI(azure_endpoint=azure_endpoint, api_version=api_version, api_key=OPENAI_API_KEY)

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

In [ ]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [ ]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {'source': document['source'], 'type': document['type']}
        return Result(page_content=f'{self.headline} \n\n {self.summary} \n\n {self.original_text}', metadata=metadata)

class Chunks(BaseModel):
    chunks: list[Chunk]


## Three steps:

1. Fetch documents from the knowledge base, like LangChain did
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in Chroma

That's it!

### Let's start with Step 1

In [ ]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob('*.md'):
            with open(file, 'r', encoding='utf-8') as fs:
                documents.append({'type': doc_type, 'source': file.as_posix(), 'file': fs.read()})
    print(f'Loaded {len(documents)} documents')
    return documents


In [ ]:
fetch_documents()